# 01 · Data, splits, corpus and metrics (Phase 1, owner: M1)

Runs Prompts 1-4 on Kaggle. Before running:
1. **Settings → Internet: On** (needs a phone-verified Kaggle account). Without internet, add Kaggle
   Datasets that contain `train.jsonl`, `shared_task_dev.jsonl`, `wiki-pages.zip` (or the unzipped
   `wiki-NNN.jsonl` files) and `hotpot_dev_distractor_v1.json`; the scripts find them under `/kaggle/input`.
2. The next cell clones the public repo `group-mauve/claim-verification-pipeline` (or pulls
   the latest commit if it is already cloned). No token or code dataset is needed.
3. Accelerator: **None** is enough for this notebook (no GPU needed; save your GPU quota).

Expected run time: about 1-1.5 hours (most of it parsing 5.4M wiki pages and the decoy BM25 search).

In [1]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

Cloning into '/kaggle/working/project'...
2a4b86d Find Kaggle inputs through symlinks; list inputs when missing
/kaggle/working/project ['.git', '.gitignore', 'README.md', 'configs', 'data', 'notebooks', 'pytest.ini', 'requirements.txt', 'src', 'tests']


In [2]:
# The official FEVER scorer's setup.py no longer builds with current pip, so instead of
# `pip install` we clone it and put its package folder (the one holding fever/scorer.py) on the path.
import os, glob, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest"], check=True)
SCORER_DIR = "/kaggle/working/fever-scorer"
if not os.path.exists(SCORER_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/sheffieldnlp/fever-scorer", SCORER_DIR], check=True)
hits = glob.glob(f"{SCORER_DIR}/**/fever/scorer.py", recursive=True)
assert hits, "fever/scorer.py not found in the cloned repo"
pkg_root = os.path.dirname(os.path.dirname(hits[0]))
os.environ["PYTHONPATH"] = pkg_root + os.pathsep + os.environ.get("PYTHONPATH", "")  # seen by !python / !pytest
sys.path.insert(0, pkg_root)
from fever.scorer import fever_score
print("official scorer importable from", pkg_root)

official scorer importable from /kaggle/working/fever-scorer/src


## Prompt 4 · metric unit tests (slide examples + official FEVER scorer)
The official-scorer test needs `shared_task_dev.jsonl`, so it is skipped on the first run and checked again after the download.

In [3]:
!python -m pytest -q -rs

.................s.......ss.                                             [100%]
=========================== short test summary info ============================
SKIPPED [1] tests/test_metrics.py:128: shared_task_dev.jsonl not found
SKIPPED [1] tests/test_retrieval.py:95: could not import 'rank_bm25': No module named 'rank_bm25'
SKIPPED [1] tests/test_retrieval.py:149: could not import 'rank_bm25': No module named 'rank_bm25'
25 passed, 3 skipped in 12.88s


## Prompt 1 · download, parse Wikipedia to Parquet, verify counts

In [4]:
!python -m src.data.download --config configs/data.yaml

06:53:38 | INFO | Downloading https://fever.ai/download/fever/train.jsonl
06:53:38 | INFO |   1 MB (3.2%) at 26.5 MB/s
06:53:38 | INFO | Saved /kaggle/working/project/data/raw/train.jsonl (33.0 MB)
06:53:38 | INFO | Downloading https://fever.ai/download/fever/shared_task_dev.jsonl
06:53:38 | INFO |   1 MB (24.1%) at 31.8 MB/s
06:53:38 | INFO | Saved /kaggle/working/project/data/raw/shared_task_dev.jsonl (4.3 MB)
06:53:38 | INFO | Downloading http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json
06:53:39 | WARNING |   failed: <urlopen error [Errno -2] Name or service not known>
06:53:42 | INFO | Loading HotpotQA distractor validation split from Hugging Face: hotpotqa/hotpot_qa
README.md: 100%|███████████████████████████| 9.52k/9.52k [00:00<00:00, 12.5MB/s]

distractor/train-00000-of-00002.parquet: downloading bytes:  | 4.07MB, 14.0kB/s 
distractor/train-00000-of-00002.parquet: downloading bytes: ▏| 40.8MB, 3.13MB/s 
distractor/train-00000-of-00002.parquet: downloading b

## Prompt 2 · train / validation / test splits

In [5]:
!python -m src.data.splits --config configs/data.yaml

06:58:35 | INFO | check ok: no claim id appears in two splits; test_5k is a subset of test_full
06:58:35 | INFO | train       30,000 claims | SUP 16,508 (55.0%) / REF 6,141 (20.5%) / NOT 7,351 (24.5%)
06:58:35 | INFO | val          2,000 claims | SUP 1,101 (55.0%) / REF 409 (20.4%) / NOT 490 (24.5%)
06:58:36 | INFO | test_full   19,998 claims | SUP 6,666 (33.3%) / REF 6,666 (33.3%) / NOT 6,666 (33.3%)
06:58:36 | INFO | test_5k      5,000 claims | SUP 1,667 (33.3%) / REF 1,667 (33.3%) / NOT 1,666 (33.3%)
06:58:36 | INFO | Example train claim: NOT ENOUGH INFO | Tyrion Lannister nearly died at least twice.
06:58:36 | INFO | Splits written to /kaggle/working/project/data/splits


## Prompt 3 · restricted corpus (~50k pages, <= 1M sentences)

In [6]:
!python -m src.data.build_corpus --config configs/data.yaml

06:58:40 | INFO | Claims: train 30,000, val 2,000, test_full 19,998
06:59:01 | INFO | Wikipedia pages available: 5,416,536
06:59:01 | INFO | Evidence pages: 9,197 (84,960 sentences)
06:59:01 | INFO | Building temporary BM25 over 5,416,536 pages (title + first sentence) ...
07:01:08 | INFO |   index built in 126s
07:01:33 | INFO |   decoys for 2,000/51,998 claims (80 claims/s)
07:01:57 | INFO |   decoys for 4,000/51,998 claims (80 claims/s)
07:02:24 | INFO |   decoys for 6,000/51,998 claims (79 claims/s)
07:02:49 | INFO |   decoys for 8,000/51,998 claims (79 claims/s)
07:03:14 | INFO |   decoys for 10,000/51,998 claims (79 claims/s)
07:03:40 | INFO |   decoys for 12,000/51,998 claims (79 claims/s)
07:04:05 | INFO |   decoys for 14,000/51,998 claims (79 claims/s)
07:04:31 | INFO |   decoys for 16,000/51,998 claims (79 claims/s)
07:04:58 | INFO |   decoys for 18,000/51,998 claims (78 claims/s)
07:05:22 | INFO |   decoys for 20,000/51,998 claims (79 claims/s)
07:05:46 | INFO |   decoys for

In [7]:
import json, pandas as pd
stats = json.load(open("data/corpus/corpus_stats.json"))
print(json.dumps(stats, indent=2))
sents = pd.read_parquet("data/corpus/sentences.parquet")
pages = pd.read_parquet("data/corpus/pages.parquet")
print(sents.shape, pages.shape)
print(pages["source"].value_counts())
sents.sample(5, random_state=42)

{
  "pages_total": 50000,
  "pages_by_source": {
    "evidence": 9197,
    "decoy": 40803
  },
  "sentences_total": 308328,
  "missing_evidence_pages_in_dump": 0,
  "coverage": {
    "train": {
      "SUPPORTS": {
        "claims": 16508,
        "all_evidence_pages_in_corpus": 1.0,
        "some_full_evidence_set_in_corpus": 1.0
      },
      "REFUTES": {
        "claims": 6141,
        "all_evidence_pages_in_corpus": 1.0,
        "some_full_evidence_set_in_corpus": 1.0
      }
    },
    "val": {
      "SUPPORTS": {
        "claims": 1101,
        "all_evidence_pages_in_corpus": 1.0,
        "some_full_evidence_set_in_corpus": 1.0
      },
      "REFUTES": {
        "claims": 409,
        "all_evidence_pages_in_corpus": 1.0,
        "some_full_evidence_set_in_corpus": 1.0
      }
    },
    "test_full": {
      "SUPPORTS": {
        "claims": 6666,
        "all_evidence_pages_in_corpus": 1.0,
        "some_full_evidence_set_in_corpus": 1.0
      },
      "REFUTES": {
        "claims

,page_id,sent_id,text,links
104895,Grant_Bluett,13,"`` Constable Frogga '' , as he is known to his...",[]
45979,California_State_Summer_School_for_the_Arts,5,"Upon completion of the program , the students ...",[]
193086,Museum_of_Science_and_Industry_-LRB-Chicago-RRB-,8,"Based on 2009 attendance , the Museum of Scien...",[Chicago]
99814,German_colonization_of_Africa,2,The Brandenburg African Company was chartered ...,"[Gold Coast (region), Ghana]"
58027,Collectors_-LRB-TV_series-RRB-,3,The panel formerly included museum curator Nic...,[Lauren Carpenter (TV personality)]


## Prompt 4 again · official scorer comparison now that the dev file exists

In [8]:
!python -m pytest -q -rs tests/test_metrics.py

...........                                                              [100%]
11 passed in 2.10s


## Package outputs for Kaggle Datasets

`fever_corpus_v1/` (about 0.5-1 GB) is what every later notebook needs: the corpus, the splits and the
small raw files. The 5.4M-page wiki Parquet (several GB) is only needed to rebuild the corpus; keep it
as a second dataset only if you have space.

After this cell: **Save Version → Save & Run All (Commit)**. When it finishes, open the notebook's
**Output** tab → **New Dataset**, name it `fever-corpus-v1`, and keep it private. Later notebooks add it
as an input at `/kaggle/input/fever-corpus-v1`.

In [9]:
import shutil, glob, os
PKG = "/kaggle/working/fever_corpus_v1"
os.makedirs(PKG, exist_ok=True)
shutil.copytree("data/corpus", f"{PKG}/corpus", dirs_exist_ok=True, ignore=shutil.ignore_patterns("_decoy_cache.json"))
shutil.copytree("data/splits", f"{PKG}/splits", dirs_exist_ok=True)
os.makedirs(f"{PKG}/raw", exist_ok=True)
for f in ["train.jsonl", "shared_task_dev.jsonl", "hotpot_dev_distractor_v1.json"]:
    shutil.copy2(f"data/raw/{f}", f"{PKG}/raw/{f}")
shutil.copy2("data/README.md", f"{PKG}/README.md")

KEEP_WIKI_PARQUET = False   # True = also keep data/wiki_sentences in the notebook output
# Delete the large files from the output so the version saves quickly.
if os.path.exists("data/raw/wiki-pages.zip"):
    os.remove("data/raw/wiki-pages.zip")
shutil.rmtree("/kaggle/working/fever-scorer", ignore_errors=True)
if not KEEP_WIKI_PARQUET:
    shutil.rmtree("data/wiki_sentences", ignore_errors=True)
!du -sh /kaggle/working/* /kaggle/working/fever_corpus_v1/*

118M	/kaggle/working/fever_corpus_v1
72K	/kaggle/working/__notebook__.ipynb
124M	/kaggle/working/project
